# Open Banking Core — entraînement et analyse

Ce notebook exécute le flux complet avec le CSV synthétique du dépôt : validation des données, split débit/crédit puis train/test, entraînement des deux branches, métriques probabilistes et diagnostics par blocs. Il utilise TF-IDF et Logistic Regression pour fonctionner sans téléchargement de modèle ni appel externe. Les scores sur ce petit jeu artificiel valident le flux technique, pas la qualité attendue sur des données réelles.

Pour ouvrir le rapport HTML interactif, installe l’extra optionnel une fois : `pip install -e .[reports]`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'pyproject.toml').is_file() and (path / 'open_banking_core').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Ouvre le notebook depuis le dépôt open-banking-core ou un de ses sous-dossiers.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from open_banking_core import CoreConfig, Evaluator, Trainer, load_transactions, split_by_sign
print(f'Dépôt : {PROJECT_ROOT}')

## 1. Charger la config et valider le CSV

Le loader mappe les colonnes du CSV vers le schéma canonique. `account_id` est ignoré et la colonne cible est conservée sous `category`.

In [ ]:
config = CoreConfig.from_yaml(PROJECT_ROOT / 'examples' / 'simple_training.yaml')
csv_path = PROJECT_ROOT / 'examples' / 'transactions.csv'
loaded = load_transactions(csv_path, config)
transactions = loaded.valid_rows
print(f'Lignes valides : {len(transactions)} | lignes rejetées : {loaded.rejected_count}')
if loaded.errors:
    display(pd.DataFrame(loaded.errors))
display(transactions.head())
display(transactions.assign(sign=transactions.amount.map(lambda value: 'debit' if value <= 0 else 'credit')).groupby('sign')['category'].value_counts().rename('count').to_frame())

## 2. Split par signe, puis train/test

Chaque signe est séparé et stratifié indépendamment. On réunit ensuite les partitions train et test pour ajuster et évaluer le `SignRouter`.

In [ ]:
sign_splits = split_by_sign(transactions, target='category', config=config.split)
split_summary = pd.DataFrame([
    {'pipeline': sign, 'train_rows': len(split.train), 'test_rows': len(split.test),
     'test_seen_description_rate': split.test_seen_description_rate}
    for sign, split in sign_splits.items()
])
display(split_summary)

train_data = pd.concat([sign_splits['debit'].train, sign_splits['credit'].train]).sort_index()
test_data = pd.concat([sign_splits['debit'].test, sign_splits['credit'].test]).sort_index()
assert set(train_data.index).isdisjoint(test_data.index)
print(f'Train : {len(train_data)} lignes | test : {len(test_data)} lignes')

## 3. Entraîner et calculer le rapport

La config d’exemple pondère certains labels pour `weighted_accuracy`. L’importance par permutation est calculée sur les lignes test, par bloc de features.

In [ ]:
router = Trainer(config).fit(train_data)
evaluation = Evaluator(config).evaluate(router, test_data, train_data=train_data)
print('Métriques globales')
display(pd.Series(evaluation.metrics, name='score').to_frame())
print('Résultats par classe')
display(pd.DataFrame(evaluation.per_class).T)

## 4. Examiner calibration, couverture, montants et importance

`ECE` utilise des bins de confiance top-label à largeur égale. La couverture mesure la part des prédictions qui atteignent chaque seuil ; l’accuracy est calculée sur cette seule part.

In [ ]:
print('Bins de confiance')
display(pd.DataFrame(evaluation.confidence_bins))
print('Couverture par seuil')
display(pd.DataFrame(evaluation.confidence_thresholds))
print('Performance par montant absolu')
display(pd.DataFrame(evaluation.amount_slices).T)
print('Importance par permutation — baisse du macro-F1')
display(pd.DataFrame(evaluation.permutation_importance).T.sort_values('mean', ascending=False))
print(f'Taux de descriptions test déjà vues au train : {evaluation.test_seen_description_rate:.1%}')

## 5. Matrices de confusion et courbes

Les matrices sont disponibles en nombres et normalisées par vraie classe. Les courbes ROC et précision-rappel sont calculées en one-vs-rest.

In [ ]:
print('Matrice de confusion — nombres')
display(pd.DataFrame(evaluation.confusion_matrix, index=evaluation.labels, columns=evaluation.labels))
print('Matrice de confusion — normalisée par ligne')
display(pd.DataFrame(evaluation.normalized_confusion_matrix, index=evaluation.labels, columns=evaluation.labels).round(3))
print('ROC AUC par classe')
display(pd.DataFrame({label: {'auc': values['auc']} for label, values in evaluation.roc_curves.items()}).T)
print('Average precision par classe')
display(pd.DataFrame({label: {'average_precision': values['average_precision']} for label, values in evaluation.precision_recall_curves.items()}).T)

## 6. Exporter le rapport HTML interactif

Le HTML autonome embarque Plotly et peut être volumineux. Pour réduire sa taille, utilise le CDN avec `include_plotlyjs=False` (connexion Internet nécessaire à l’ouverture).

In [ ]:
html_path = PROJECT_ROOT / 'runs' / 'evaluation-report.html'
try:
    evaluation.write_html(html_path)
    print(f'Rapport HTML écrit : {html_path}')
    from IPython.display import IFrame
    relative_html = html_path.relative_to(PROJECT_ROOT).as_posix()
    display(IFrame(src=f'../{relative_html}', width='100%', height=900))
except ImportError as error:
    print(error)
    print('Installe Plotly avec : pip install -e .[reports]')